# Solutions · 14 · Robustness and fundamental limitations

Worked solutions to the exercises of [notebook 14](../notebooks/14_robustness_and_limitations.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, robust as rb, frequency as fr, design as dg, polynomials as pl, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

For the levitated ball (unstable pole at 44 rad/s) design a lead controller with crossovers of 50, 100 and 200 rad/s and tabulate Ms, the margins and the peak |CS|. Confirm that the crossover must exceed about 2p for a decent Ms, and read the price in noise amplification.

In [2]:
ml = models.magnetic_levitation()
P = ml["system"].to_tf()
P = sy.tf(-P.num, P.den)                      # the current's effect on the gap is negative: flip the input sign so that the plant has positive gain (phase -180)
p_un = ml["unstable_pole"]
rows = []
for wc in (50.0, 100.0, 200.0):
    d = dg.design_lead(P, wc, 45.0)
    L = sy.series(d["controller"], P)
    g4 = rb.gang_of_four(P, d["controller"])
    w = np.logspace(0, 4, 3000)
    stable = bool(np.all(sy.feedback(L).poles().real < 0))
    rows.append((wc, wc / p_un, stable, rb.sensitivity_peak(L)["Ms"] if stable else np.nan, d["margins"]["phase_margin"], np.max(np.abs(fr.freqresp(g4["CS"], w)))))
print(pd.DataFrame(rows, columns=["crossover", "w_c / p", "closed loop stable", "Ms", "PM (deg)", "peak |CS|"]).to_string(index=False, float_format="%.2f"))

 crossover  w_c / p  closed loop stable   Ms  PM (deg)  peak |CS|
     50.00     1.13               False  NaN     38.23     453.94
    100.00     2.26                True 1.37     50.00    1173.11
    200.00     4.52                True 1.35     50.00    4114.03


An unstable pole at p demands a crossover above about 2p: at ω_c = 50 (just above p = 44) the lead design
does not even stabilise the loop - the Nyquist curve fails to encircle −1 - while at 100 and 200 the loop is
comfortable (M_s = 1.4) - and the price shows in the peak of CS, the noise amplification, which quadruples
each time the crossover doubles. Unstable plants force bandwidth, and bandwidth costs noise: a levitation
sensor must be clean. (The plant's input gain is negative; the sign is flipped so that the design works with
the usual −180° convention, which `design_lead` expects.)

## Exercise 2

The plant (2 - s)/(s + 1)^2 under a PI controller: compute Ms as a function of the controller gain K and find the gain that minimises it; compare the corresponding crossover with the z/2 rule.

In [3]:
Pz = sy.tf([-1.0, 2.0], [1.0, 2.0, 1.0])
rows = []
for K in np.linspace(0.1, 1.5, 15):
    C = sy.tf([K, 0.5 * K], [1.0, 0.0])
    L = sy.series(C, Pz)
    m = fr.margins(L)
    rows.append((K, rb.sensitivity_peak(L)["Ms"], m["w_gc"]))
rows = np.array(rows)
print("Ms rises monotonically with the gain:", np.round(rows[:, 1], 2))
i16 = np.argmax(rows[:, 1] > 1.6)
print(f"a design target Ms <= 1.6 allows K up to about {rows[i16 - 1, 0]:.2f}, where the crossover is {rows[i16 - 1, 2]:.3f} rad/s = z/{2.0 / rows[i16 - 1, 2]:.1f}; at the z/2 crossover (K = {rows[np.argmin(np.abs(rows[:, 2] - 1.0)), 0]:.2f}) Ms is already {rows[np.argmin(np.abs(rows[:, 2] - 1.0)), 1]:.2f}")

Ms rises monotonically with the gain: [ 1.11  1.23  1.37  1.53  1.72  1.96  2.25  2.61  3.09  3.75  4.71  6.25
  9.09 16.12 63.19]
a design target Ms <= 1.6 allows K up to about 0.40, where the crossover is 0.460 rad/s = z/4.3; at the z/2 crossover (K = 0.80) Ms is already 2.61


There is no interior optimum: M_s rises monotonically with the gain, because every increase pushes the
crossover towards the zero whose phase lag drives the curve towards −1. The z/2 rule is therefore a ceiling,
not a target - at a crossover of z/2 the loop already has M_s = 2.6 - and a robust design with M_s ≤ 1.6
sits at about a quarter of the zero. Right-half-plane zeros cost more bandwidth than the rule of thumb
suggests.

## Exercise 3

**Implement it yourself:** verify Bode's integral numerically for the stabilised unstable loop of the notebook by integrating ln|S| on a fine grid up to 10^5 rad/s, and show that truncating the integral at 100 rad/s misses a visible fraction (the tail decays slowly).

In [4]:
L = sy.tf([30.0, 60.0], pl.from_roots([1, -5, -8]))
for w_max in (1e2, 1e3, 1e5):
    bi = rb.bode_sensitivity_integral(L, w_max=w_max, n=400000)
    print(f"integral of ln|S| up to {w_max:.0e} rad/s: {bi['integral']:.4f} (theory pi = {np.pi:.4f}, missing {100 * (1 - bi['integral'] / np.pi):.1f} %)")

integral of ln|S| up to 1e+02 rad/s: 2.8423 (theory pi = 3.1416, missing 9.5 %)
integral of ln|S| up to 1e+03 rad/s: 3.1115 (theory pi = 3.1416, missing 1.0 %)
integral of ln|S| up to 1e+05 rad/s: 3.1412 (theory pi = 3.1416, missing 0.0 %)


ln|S| decays like −|L| at high frequency, and |L| of a relative-degree-2 loop decays like 1/ω²: the tail of
the integral is of order 1/ω_max and a cut-off at 100 rad/s misses a visible fraction. The integral is an
exact identity only over the whole axis - a numerical check must go far enough out.